In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

import torchvision
import torchvision.transforms as transforms

# Set device to MPS if available, otherwise fallback to CPU
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

Using device: mps


In [2]:
transform_train = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) # Standard ImageNet stats
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) # Standard ImageNet stats
])

In [3]:
train_data = torchvision.datasets.CIFAR10(root='./data', train=True, transform=transform_train, download=True)
test_data = torchvision.datasets.CIFAR10(root='./data', train=False, transform=transform_test, download=True)

train_loader = torch.utils.data.DataLoader(train_data, batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
test_loader = torch.utils.data.DataLoader(test_data, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)


/Users/allen_ou/Documents/code/csi5140_project_1/project_1_scratcg/.venv/lib/python3.14/site-packages/torchvision/datasets/cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


In [4]:
image, label = train_data[0]
image.size()

torch.Size([3, 32, 32])

In [5]:
class_names = ['plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']

In [6]:
class TheNetwork(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv2d(3, 12, 5) #shape in-kernel / stride +1 = (12,28,28)
        self.pool = nn.MaxPool2d(2, 2) #2x2 pixels down to 1 px (12, 14, 14)
        self.conv2 = nn.Conv2d(12, 24, 5) #(24, 10, 10) -> (24, 5, 5) -> flatten > 120
        self.ann1 = nn.Linear(24 * 5 * 5, 120)
       # self.ann2 = nn.Linear(120, 84)
       # self.ann3 = nn.Linear(84, 50)
        self.ann4 = nn.Linear(120, 10)
    
    def forward(self, x):
        x= self.pool(F.relu(self.conv1(x)))
        x= self.pool(F.relu(self.conv2(x)))
        x= torch.flatten(x, 1)
        x= F.relu(self.ann1(x))
        x= self.ann4(x)

        return x


In [7]:
nnet = TheNetwork().to(device)
loss_function = nn.CrossEntropyLoss()
optimz = optim.Adam(nnet.parameters(), lr=0.0001)

In [8]:
#train
for epoch in range(50):
    
    running_loss = 0.0

    for i, (inputs, labels) in enumerate(train_loader):
        
        inputs = inputs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        optimz.zero_grad()

        out = nnet(inputs)

        loss = loss_function(out, labels)
        loss.backward()
        optimz.step()

        running_loss += loss.item()
    print(f'Loss: {running_loss / len(train_loader):.4f}')

/Users/allen_ou/Documents/code/csi5140_project_1/project_1_scratcg/.venv/lib/python3.14/site-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Loss: 1.8196
Loss: 1.5341
Loss: 1.4413
Loss: 1.3807
Loss: 1.3401
Loss: 1.3009
Loss: 1.2676
Loss: 1.2389
Loss: 1.2152
Loss: 1.1909
Loss: 1.1717
Loss: 1.1519
Loss: 1.1336
Loss: 1.1131
Loss: 1.0976
Loss: 1.0850
Loss: 1.0760
Loss: 1.0592
Loss: 1.0475
Loss: 1.0371
Loss: 1.0233
Loss: 1.0147
Loss: 1.0044
Loss: 0.9925
Loss: 0.9866
Loss: 0.9744
Loss: 0.9686
Loss: 0.9570
Loss: 0.9518
Loss: 0.9381
Loss: 0.9316
Loss: 0.9244
Loss: 0.9149
Loss: 0.9098
Loss: 0.9033
Loss: 0.8971
Loss: 0.8920
Loss: 0.8831
Loss: 0.8803
Loss: 0.8736
Loss: 0.8661
Loss: 0.8625
Loss: 0.8554
Loss: 0.8479
Loss: 0.8451
Loss: 0.8388
Loss: 0.8383
Loss: 0.8287
Loss: 0.8288
Loss: 0.8229


In [9]:
#save/load weights 
torch.save(nnet.state_dict(), 'trained_nn.pth')
#net2 = TheNetwork()
#net2.load_state_dict(torch.load('trained_nn.pth'))

In [10]:
#accuracy
correct = 0
total = 0

nnet.eval()

with torch.no_grad():
    for i, (inputs, labels) in enumerate(test_loader):
        inputs = inputs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        outputs = nnet(inputs)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = (correct/total) * 100
print(f'Accuracy: {accuracy}%')
        

Accuracy: 69.69999999999999%
